langgraph初探，不用真实的llm模型，使用一个简单的模拟器来展示langgraph的基本功能和工作流程。

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END

def mock_llm(state: MessagesState):
    return {"messages": [{"role": "ai", "content": "hello world"}]}

graph = StateGraph(MessagesState)
graph.add_node(mock_llm)
graph.add_edge(START, "mock_llm")
graph.add_edge("mock_llm", END)
graph = graph.compile()

graph.invoke({"messages": [{"role": "user", "content": "hi!"}]})

接下来使用真实的LLM来尝试，我们这里不调用本地LLM而是使用云端LLM，通过配置api-key访问，这里以openai schema为例

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import HumanMessage
import os

# 实例化 LLM — 支持任何 OpenAI 兼容 API
# 推荐 2026 年开发模型：DeepSeek-V4（性价比最高）、Qwen3（开源首选）
llm = ChatOpenAI(
    model="deepseek-chat",               # DeepSeek V4，$0.87/1M output
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
    temperature=0,
)

# 也可用本地 Ollama：
# llm = ChatOpenAI(
#     model="qwen3:14b",
#     base_url="http://localhost:11434/v1",
#     api_key="none",
# )

def call_llm(state: MessagesState):
    # 调用 LLM 模型
    response = llm.invoke(state["messages"])
    return {"messages": [response]}  # MessagesState 自动 add_messages，只需返回本身即可


# 创建状态图
graph = StateGraph(MessagesState)
graph.add_node("call_llm", call_llm)
graph.add_edge(START, "call_llm")
graph.add_edge("call_llm", END)

# 编译图
compiled_graph = graph.compile()

greet_state = {"messages": [HumanMessage(content="hi! I am studying langgraph.")]}
response = compiled_graph.invoke(greet_state)
response["messages"]

下面我们探究一下langgraph中的状态机制，可以发现返回的内容会进行部分字段的消息更新，也就是合并而不是替换，这种类似于giff的写法容易保留和回溯，同时配合annotate注解和reducer函数，完成消息的自动累加而不是简单的替换

In [ ]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END

class TestState(TypedDict):
    messages: Annotated[list, add]
    counter: int
    data: str

def node_only_messages(state: TestState):
    print(f"Node1 收到: {state}")
    return {"messages": ["msg from node1"]}

def node_only_counter(state: TestState):
    print(f"Node2 收到: {state}")
    return {"counter": 1}

def node_only_data(state: TestState):
    print(f"Node3 收到: {state}")
    return {"data": "hello"}

def node_no_return(state: TestState):
    print(f"Node4 收到: {state}")
    return {}  # 返回空字典，不更新任何字段

def node_append_messages(state: TestState):
    print(f"Node5 收到: {state}")
    return {"messages": ["msg from node5, it should be appended to messages, not replace"]}

# 构建图
graph = StateGraph(TestState)
graph.add_node("n1", node_only_messages)
graph.add_node("n2", node_only_counter)
graph.add_node("n3", node_only_data)
graph.add_node("n4", node_no_return)
graph.add_node("n5", node_append_messages)

graph.add_edge(START, "n1")
graph.add_edge("n1", "n2")
graph.add_edge("n2", "n3")
graph.add_edge("n3", "n4")
graph.add_edge("n4", "n5")
graph.add_edge("n5", END)

compiled = graph.compile()

# 执行
result = compiled.invoke({"messages": [], "counter": 0, "data": ""})
print("最终状态:", result)


以上展示了langGraph的语法机制，大概可以总结为：
1. langGraph本质是用字典进行状态的封装，尽管写为`class`，但是使用`TypedDict`关键字表示这是一个字典，访问的时候要用`["字段名"]`而不是`.字段名`
1. 一旦你用 `graph = StateGraph(mystate)` 绑定了状态类型，这个状态会成为图中所有节点的统一接口标准;
2. 节点函数不需要返回完整的状态，只需要返回要更新的字段；
3. 配合`Annotate`注解和`reducer`函数，能够完成字段的的自动累加而不是简单的替换，例如`MessagesState`的默认实现
4. 用`HumanMessage`等langGraph包装好的消息体，这可以完成消息的自动处理和转换而不用考虑不同schema之间的转换，并且能够很好的区分system、human、ai、tool等不同的消息

刚刚我们构建的一些操作，本质是链式的, 但是这些功能理论上langchain等链式架构也可以轻易完成，接下来探究一下langGraph真正的核心机制，状态图，
首先可以通过代码来看到图的结构, 这里我们使用 `get_graph()`函数，注意，一定是`compile`之后的图

In [ ]:
from IPython.display import Image

# 直接在 Notebook 中显示图片
Image(compiled.get_graph().draw_mermaid_png())

可以看到整个工作流是线性的，但是如果要构建一个良好的agent，需要进行的处理绝对不是简单的线性的，需要根据每一步执行的结果进行调整下一步要执行的操作，一个很简单的例子，如果我们想要LLM输出一个结构化的Json并给出了schema，但是LLM的输出总是没有那么稳定，我们需要一次后检判断LLM的输出是否非法，如果非法需要让他重新生成，并且告诉LLM为什么错了，这就是一个标准的回溯逻辑。

我们引入langGraph的一个核心成员，条件边，也就是`conditionnal_edge`, 通过一个字节的返回值来指定会走那条分支，值得注意的是，分支函数不是节点函数，不需要返回一个字段，而是返回一个字符串字面量，字符串字面量用于指示下一个节点的名字，标准的写法如下：
```python

def router_function(state: State) -> Literal["node_a", "node_b"]:
    if condition:
        return "node_a"
    else:
        return "node_b"

# 1. 使用函数返回值决定下一个节点
graph.add_conditional_edges(
    source_node,      # 起始节点
    router_function,  # 路由函数，返回节点名称字符串
    path_map          # 可选：路径映射字典，如果不传入这个参数，需要保证条件边函数的返回值的字面量有对应的节点函数
)

# 2. 返回多个节点（并行执行）
graph.add_conditional_edges(
    source_node,
    router_function,  # 返回 List[str]
    path_map
)
```

path_map 的作用：
1. 解耦：让路由函数返回抽象状态（如 "high"/"medium"/"low"），而不是具体的节点名
2. 复用：同一个路由函数可用于不同的图，通过映射适配不同的节点名
3. 可读性：用有意义的抽象返回值，而不是硬编码节点名

In [ ]:
from typing import TypedDict, Literal
from IPython.display import Image
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage

class State(TypedDict):
    messages: list
    next_step: str  # 控制流程的字段

# 定义路由函数
def decide_next(state: State) -> Literal["continue", "end"]:
    """根据最后一条消息决定下一步"""
    last_msg = state["messages"][-1]
    
    if "再见" in last_msg.content:
        return "end"      # 返回节点名称
    else:
        return "continue" # 返回节点名称

# 构建图
builder = StateGraph(State)

# 添加节点
def process(state: State):
    print("处理中...")
    return {"messages": [AIMessage("继续对话")]}

def finish(state: State):
    print("结束对话")
    return {"messages": [AIMessage("再见！")]}

builder.add_node("process", process)
builder.add_node("finish", finish)

# 设置起始边
builder.add_edge(START, "process")

# 添加条件边：从 process 节点出发
builder.add_conditional_edges(
    "process",           # 源节点
    decide_next,         # 路由函数
    {
        "continue": "process",  # 返回 "continue" 时去 process
        "end": "finish"         # 返回 "end" 时去 finish
    }
)

# 终止边
builder.add_edge("finish", END)

graph = builder.compile()

# 测试
Image(graph.get_graph().draw_mermaid_png())

上面只是构建了一个分支路，但是我们仔细分析后会发现，任意传入一串信息，如果不是“再见”这条消息，就会出发状态机的死循环，会诱发底层的`GraphRecursionError  `

In [ ]:
result = graph.invoke({"messages": [HumanMessage("你好")], "next_step": ""})

这个时候我们就需要重试的兜底，一般有两种处理逻辑
第一是在状态中记录重试次数，如果发现在不停重试，就进行兜底处理或者直接结束
第二个是人工干预节点，也就是human-in-the-loop，这也是langGraph的一个重要成员，我们可以在执行结束之后插入一个人工干预节点

Human-in-the-Loop (HITL) 是一种设计模式，允许在 AI 工作流的特定节点暂停执行，等待人工审查、修改或批准后，再从中断点恢复。

LangGraph 通过持久化检查点和中断机制优雅地实现了 HITL。其核心原理是：工作流每执行一步，系统都会自动保存当前状态的"快照"（就像游戏存档）。我们可以配置在某些节点前或后设置断点，执行流到达时便会暂停，等待人工干预。人类可以查看状态、甚至修改参数，然后通过恢复命令让工作流从断点继续运行

LangGraph 主要支持两种中断方式，分别适用于不同的控制需求：

1. 静态中断：在固定节点前/后暂停
这种方式在编译工作流图时就声明好中断点，适合标准化的审核流程（如"每次调用工具前都要我批准"）。

核心参数：`interrupt_before=["节点名"]` 或 `interrupt_after=["节点名"]`

实现步骤：

编译图时，通过`interrupt_before`指定在哪里暂停。
使用`app.stream()`运行（而不是 invoke），因为`stream`遇到中断会平静地返回，而`invoke`会直接抛出异常。
中断后，通过`app.get_state(config)`查看当前状态（如模型想调用的工具及其参数）。
人工决策后，使用`Command(resume=True)`恢复执行。
2. 动态中断：在节点内部条件触发
这种方式更灵活，适合需要根据业务逻辑动态决定是否中断的场景（例如：只有检测到敏感内容时才请求人工复核）。

核心工具：`interrupt()` 函数

实现方式：在某个节点函数内部，当满足特定条件时，调用 `interrupt()` 暂停工作流并等待人工输入。

最简单的实现方式是，在process节点中加入一个`interrupt()`即可，当然为了需要在状态中加入一个loop_num来进行循环次数的记录，我们重写一下刚刚的代码

In [ ]:
from typing import TypedDict, Literal, Annotated
from IPython.display import Image
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.types import interrupt
from langchain_core.messages import HumanMessage, AIMessage, AnyMessage

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    if_end: bool  # 控制流程的字段
    loop_num: int # 添加一个字段来测试状态的更新

# 定义路由函数
def decide_next(state: State) -> Literal["continue", "end"]:
    """根据最后一条消息决定下一步"""
    if_end = state["if_end"]

    if if_end is False:
        return "continue"
    elif if_end is True:
        return "end"

# 构建图
builder = StateGraph(State)

# 添加节点
def process(state):
    new_loop_num = state["loop_num"] + 1

    if new_loop_num >= 3:
        decision = interrupt("是否继续？输入 y 继续, 否则结束")
        return {
            "loop_num": new_loop_num,
            "if_end": decision.lower() != "y",
            "messages": [AIMessage(content="继续对话")]
        }

    return {
        "loop_num": new_loop_num,
        "if_end": False,
        "messages": [AIMessage(content="继续对话")]
    }
def finish(state: State):
    print("结束对话")
    return {"messages": [AIMessage("再见！")]}

builder.add_node("process", process)
builder.add_node("finish", finish)

# 设置起始边
builder.add_edge(START, "process")

# 添加条件边：从 process 节点出发
builder.add_conditional_edges(
    "process",           # 源节点
    decide_next,         # 路由函数
    {
        "continue": "process",  # 返回 "continue" 时去 process
        "end": "finish"         # 返回 "end" 时去 finish
    }
)

# 终止边
builder.add_edge("finish", END)

graph = builder.compile()

# 测试
Image(graph.get_graph().draw_mermaid_png())

In [ ]:
init_state = {"messages": [HumanMessage("你好")], "approved": True, "loop_num": 0}
result = graph.invoke(init_state)
print("最终状态:", result)
# 可以看到，在最终状态中出现了一个'__interrupt__': [Interrupt(value='是否继续？输入‘y’继续，否则结束'，表示进程被挂起了，等待用户输入决定下一步的走向）

我们需要通过`resume`传入用户的输入，但是`resume`必须搭配`checkpoint`来使用，最常用的是`MemorySaver`和`SqliteSaver`，并且通过一个config来指定

**Checkpointer：存储机制本身**
职责：定义"怎么存"
```python
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.checkpoint.postgres import PostgresSaver

# 不同实现代表不同的存储策略
memory_checkpointer = InMemorySaver()      # 存在内存（程序重启就丢）
sqlite_checkpointer = SqliteSaver("data.db")  # 存在本地文件
postgres_checkpointer = PostgresSaver(...)    # 存在数据库
```
Checkpointer 负责：
序列化：把图的状态（变量、执行位置等）转换成可存储的格式
持久化：实际写入到内存/文件/数据库
反序列化：恢复时把存储的数据还原成状态对象
版本管理：处理同一个 thread_id 的多次更新

**Config：寻址信息 + 执行上下文**
职责：告诉系统"存到哪里"和"为谁存"
Config 负责：
唯一标识：thread_id 区分不同的执行会话
版本定位：可选的 checkpoint_id 指定恢复到历史状态（类似 Git 的 commit hash）
运行配置：还可以包含其他运行时参数（如超时、重试策略等）


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
# 1. 准备阶段
checkpointer = InMemorySaver()  # 决定"怎么存"
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "my_session"}}  # 决定"存到哪"

init_state = {"messages": [HumanMessage("你好")], "if_end": False, "loop_num": 0}

# 2. 执行中断
for event in graph.stream(init_state, config):
    print("事件:", event)
    if "__interrupt__" in event:
        # 此时发生：
        # - graph 调用 checkpointer.save(thread_id="my_session", state=当前状态)
        # - checkpointer 将状态序列化后存入内存/数据库
        pass

user_input = input("请输入是否继续（y/n）: ")
# 3. 恢复执行
# - config 提供 thread_id，告诉系统去读哪个会话
# - checkpointer 根据 thread_id 找到并反序列化出状态
for resume_event in graph.stream(Command(resume=user_input), config):
    pass

刚刚我们用一次for循环处理了一个中断，但是单次执行流中更多情况下会出现多次中断，这种情况下，更通用的方法是用一个while循环

In [13]:
# 可以处理任意次中断
def run_with_interactive_interrupts(graph, initial_state, config):
    """持续处理中断，直到图执行完成"""
    
    # 开始或恢复执行
    events = graph.stream(initial_state, config)
    
    while True:
        try:
            for event in events:
                print("事件:", event)
                
                # 检查是否有中断
                if "__interrupt__" in event:
                    # 获取中断信息
                    interrupt_info = event["__interrupt__"]
                    print(f"中断原因: {interrupt_info}")
                    
                    # 等待用户输入
                    user_input = input(">>> 请输入: ")
                    
                    # 用 Command 恢复，继续循环
                    events = graph.stream(Command(resume=user_input), config)
                    break  # 跳出内层 for，继续外层 while
            else:
                # 没有中断，正常完成
                break
                
        except StopIteration:
            break



In [14]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
# 1. 准备阶段
checkpointer = InMemorySaver()  # 决定"怎么存"
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "my_session"}}  # 决定"存到哪"

init_state = {"messages": [HumanMessage("你好")], "if_end": False, "loop_num": 0}

# 2. 执行，自动处理中断
run_with_interactive_interrupts(graph, init_state, config)

事件: {'process': {'loop_num': 1, 'if_end': False, 'messages': [AIMessage(content='继续对话', additional_kwargs={}, response_metadata={}, id='1a491806-dea7-452b-b6d3-746e52d84fcd', tool_calls=[], invalid_tool_calls=[])]}}
事件: {'process': {'loop_num': 2, 'if_end': False, 'messages': [AIMessage(content='继续对话', additional_kwargs={}, response_metadata={}, id='adcb543c-0e70-4e8c-b9ff-221718b8d5a1', tool_calls=[], invalid_tool_calls=[])]}}
事件: {'__interrupt__': (Interrupt(value='是否继续？输入 y 继续, 否则结束', id='5c3bf88e40f6c1abd15b6012c56ca6b6'),)}
中断原因: (Interrupt(value='是否继续？输入 y 继续, 否则结束', id='5c3bf88e40f6c1abd15b6012c56ca6b6'),)


事件: {'process': {'loop_num': 3, 'if_end': False, 'messages': [AIMessage(content='继续对话', additional_kwargs={}, response_metadata={}, id='51a9504a-9223-4d63-b510-32818fecb4b9', tool_calls=[], invalid_tool_calls=[])]}}
事件: {'__interrupt__': (Interrupt(value='是否继续？输入 y 继续, 否则结束', id='3d25d17d41faac743b6b3924b2843441'),)}
中断原因: (Interrupt(value='是否继续？输入 y 继续, 否则结束', id='3d25d17d41faac743b6b3924b2843441'),)
事件: {'process': {'loop_num': 4, 'if_end': False, 'messages': [AIMessage(content='继续对话', additional_kwargs={}, response_metadata={}, id='aaf8f5c2-d7b5-4cbe-9153-ed8a172320d9', tool_calls=[], invalid_tool_calls=[])]}}
事件: {'__interrupt__': (Interrupt(value='是否继续？输入 y 继续, 否则结束', id='43bcd1abea6c72c2ec476836a3c3afc7'),)}
中断原因: (Interrupt(value='是否继续？输入 y 继续, 否则结束', id='43bcd1abea6c72c2ec476836a3c3afc7'),)
事件: {'process': {'loop_num': 5, 'if_end': True, 'messages': [AIMessage(content='继续对话', additional_kwargs={}, response_metadata={}, id='a58352b9-c95c-4706-a065-88a32f3b898f', tool_calls=[], i

### LangGraph 1.2+ 新增：create_react_agent 高级 API

上面我们手动构建了完整的 ReAct Agent。LangGraph 1.2+ 提供了 `create_react_agent` 函数，一行代码就能创建标准的 ReAct Agent。

此外，LangGraph 2.0（2026年4月发布）引入了新的**状态机范式**用于多智能体编排，LangChain 1.0 统一使用 `create_agent()` 作为标准入口：

```python
from langgraph.prebuilt import create_react_agent

# 一行创建 ReAct Agent（自动处理 LLM ↔ Tools 循环）
agent = create_react_agent(
    model=llm,
    tools=[tool1, tool2, tool3],
    prompt="你是一个有帮助的助手",
    checkpointer=InMemorySaver(),
)

# 直接调用
result = agent.invoke({"messages": [HumanMessage("你好")]})
```

`create_react_agent` 自动处理了：LLM → 工具调用循环、工具结果回传、条件路由、错误处理和兜底。

**什么时候用 `create_react_agent`，什么时候手动构建 `StateGraph`？**

| 场景 | 推荐 |
|------|------|
| 标准 ReAct Agent | `create_react_agent` |
| 需要自定义节点/路由 | 手动 `StateGraph` |
| 需要 Human-in-the-Loop | 手动 `StateGraph` |
| 需要复杂的多 Agent 协作 | LangGraph 2.0 或手动 `StateGraph` |

本教程后面会深入手动构建 `StateGraph`，这样你才能真正理解 Agent 的内部机制。

> **2026年 LangGraph 1.2 新特性**：Per-Node Timeouts、Node-Level Error Handlers（支持 Saga 补偿事务）、Graceful Shutdown（零丢失滚动部署）、DeltaChannel（长会话存储减少 41x）。

### LangGraph 1.2+ 新增：create_react_agent 高级 API

上面我们手动构建了完整的 ReAct Agent。LangGraph 1.2+ 提供了一个更简洁的 `create_react_agent` 函数，一行代码就能创建标准的 ReAct Agent：

```python
from langgraph.prebuilt import create_react_agent

# 一行创建 ReAct Agent（自动处理 LLM ↔ Tools 循环）
agent = create_react_agent(
    model=llm,
    tools=[tool1, tool2, tool3],  # LangChain Tool 列表
    prompt="你是一个有帮助的助手",  # 可选 system prompt
    checkpointer=InMemorySaver(), # 可选，启用状态持久化
)

# 直接调用
result = agent.invoke({"messages": [HumanMessage("你好")]})
```

`create_react_agent` 自动处理了：
- LLM → 工具调用的循环
- 工具结果回传给 LLM
- 条件路由判断
- 错误处理和兜底

**什么时候用 `create_react_agent`，什么时候手动构建 `StateGraph`？**

| 场景 | 推荐 |
|------|------|
| 标准 ReAct Agent | `create_react_agent` |
| 需要自定义节点/路由 | 手动 `StateGraph` |
| 需要 Human-in-the-Loop | 手动 `StateGraph` |
| 需要复杂的多 Agent 协作 | 手动 `StateGraph` |

本教程后面会深入手动构建 `StateGraph`，这样你才能真正理解 Agent 的内部机制。

In [ ]:
from typing import Literal, Annotated, TypedDict
import os
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage

# 实例化 LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0,
)


# ========== 1. 定义状态 ==========
class State(TypedDict):
    messages: Annotated[list, add_messages]  # 对话历史
    user_name: str                           # 用户姓名
    user_issue: str                          # 用户问题
    need_confirmation: bool                  # 是否需要确认
    confirmation_result: bool                # 确认结果（True=确认，False=修改）

# ========== 2. 定义节点 ==========

def collect_info(state: State):
    """收集用户信息（姓名 + 问题）"""
    # 检查是否已有姓名
    if not state.get("user_name"):
        # 中断并询问姓名
        name = interrupt("请告诉我您的姓名：")
        return {"user_name": name}

    # 检查是否已有问题
    if not state.get("user_issue"):
        issue = interrupt("请描述您遇到的问题：")
        return {"user_issue": issue}

    return {}

def summarize_issue(state: State):
    """汇总用户信息，并调用大模型生成总结"""
    prompt = f"""
    用户姓名：{state['user_name']}
    用户问题：{state['user_issue']}
    请用礼貌的语气总结用户的问题，并询问用户确认是否正确。
    """

    response = llm.invoke(prompt)

    return {
        "messages": [AIMessage(content=response.content)],
        "need_confirmation": True  # 标记需要确认
    }

def handle_confirmation(state: State):
    """处理用户确认（中断等待用户确认或修改）"""

    # 中断并询问确认
    decision = interrupt(
        "请确认以上信息是否正确？\n"
        "输入 'y' 确认，输入 'n' 修改，输入其他内容说明修改原因："
    )

    if decision.lower() == 'y':
        return {"confirmation_result": True, "need_confirmation": False}
    else:
        # 用户要修改，记录修改原因
        return {
            "confirmation_result": False,
            "need_confirmation": True,  # 仍然需要确认
            "messages": [AIMessage(content=f"好的，我将根据您的意见修改：{decision}")]
        }

def process_issue(state: State):
    """处理用户问题（确认后执行）"""

    prompt = f"""
    用户姓名：{state['user_name']}
    用户问题：{state['user_issue']}

    请提供详细的解决方案：
    """

    response = llm.invoke(prompt)
    return {"messages": [AIMessage(content=response.content)]}

# ========== 3. 定义条件边 ==========

def need_collect_info(state: State) -> Literal["collect_info", "summarize"]:
    """判断是否需要继续收集信息"""
    if not state.get("user_name") or not state.get("user_issue"):
        return "collect_info"
    return "summarize"

def after_summary(state: State) -> Literal["handle_confirmation", "process_issue"]:
    """判断总结后进入确认还是直接处理"""
    if state.get("need_confirmation", False):
        return "handle_confirmation"
    return "process_issue"

def after_confirmation(state: State) -> Literal["collect_info", "process_issue", "summarize"]:
    """根据确认结果决定下一步"""
    if state.get("confirmation_result", False):
        return "process_issue"
    return "collect_info"

# ========== 4. 构建图 ==========

builder = StateGraph(State)

# 添加节点
builder.add_node("collect_info", collect_info)
builder.add_node("summarize", summarize_issue)
builder.add_node("handle_confirmation", handle_confirmation)
builder.add_node("process_issue", process_issue)

# 设置入口点
builder.add_edge(START, "collect_info")

# 添加条件边
builder.add_conditional_edges("collect_info", need_collect_info)
builder.add_conditional_edges("summarize", after_summary)
builder.add_conditional_edges("handle_confirmation", after_confirmation)

# 添加普通边
builder.add_edge("process_issue", END)

# 编译图（必须带 checkpointer）
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

# ========== 5. 多轮交互执行器 ==========

def run_with_interrupts(graph, config, initial_state=None):
    """持续处理中断，支持多轮对话"""

    events = graph.stream(initial_state or {}, config)

    while True:
        interrupted = False

        for event in events:
            if "__interrupt__" in event:
                # 获取中断信息
                interrupt_info = event["__interrupt__"]
                print(f"\n中断等待输入：")
                print(f"   {interrupt_info[0].value}")

                # 等待用户输入
                user_input = input("\n  您的输入: ")

                # 恢复执行
                events = graph.stream(Command(resume=user_input), config)
                interrupted = True
                break

        if not interrupted:
            print("\n 对话完成！")
            break

# ========== 6. 执行示例 ==========

if __name__ == "__main__":
    # 配置唯一的会话ID
    config = {"configurable": {"thread_id": "customer_service_001"}}

    # 开始多轮对话
    print("=" * 50)
    print("智能客服系统启动")
    print("=" * 50)

    run_with_interrupts(graph, config, {"messages": [], "need_confirmation": False})